In [1]:
"""
This generates conversational datasets for each model - this is used for evaluating validity of role probes. This takes user 
messages from toxicchat/oasst, then generates model-specific assistant responses for each model using Openrouter. Allows for
multiturn conversations. Runs models locally as a fallback if unavailable via API.
"""
None

In [2]:
"""
Imports
"""
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import os
from tqdm import tqdm

from utils.openrouter import get_openrouter_responses

seed = 1234

load_dotenv('.env')
std_params = {'temperature': 1, 'top_p': 1, 'top_k': 0} # Most reasoning models require 1 temp
ws = '/root/prompt-injection-as-role-confusion'

# Import conversations + retain only user messages

In [3]:
# Dataset preparation uses the original sampling functions; cached below.

In [4]:
# OASST user turns were prepared with the original tree traversal and filters.

In [5]:
user_queries_df = pd.read_csv('outputs/local-gptoss20b/user_queries.csv')
display(user_queries_df.groupby('dataset').agg(conversations=('conv_id','nunique')))

,conversations
dataset,
oasst,100
toxicchat,100


# Generate assistant/cot responses by model

In [6]:
target_models = [{'model':'openai/gpt-oss-20b','model_prefix':'gptoss-20b'}]

In [7]:
print('OpenRouter generation skipped: using the local checkpoint below.')

OpenRouter generation skipped: using the local checkpoint below.


# Local fallback for unavailable models

In [8]:
"""
All below code is a fallback for models which do NOT have available endpoints. Avoid if possible.
"""
import torch

from local_runtime import load_model_and_tokenizer
from utils.memory import check_memory

main_device = 'cuda:0'

model_prefix = 'gptoss-20b'
tokenizer, model, model_architecture, model_n_layers = load_model_and_tokenizer(model_prefix, device = main_device)

check_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Loaded local 20B: MXFP4/Triton experts, BF16 activations, eager attention


Device 0: NVIDIA A100-SXM4-80GB
  Allocated: 12.83 GB
  Reserved: 12.84 GB
  Total: 79.25 GB



In [9]:
"""
Run generations
"""
MAX_INPUT_TOKENS = 1024 * 12
MAX_NEW_TOKENS = 4_000
TEMPERATURE = 1.0
BATCH_SIZE = 64

def chunk_list(lst, n):
    return [lst[:16]] + [lst[i:i+n] for i in range(16,len(lst),n)]

@torch.inference_mode()
def generate_local_batch(message_histories):
    """
    Params:
        @message_histories: list[list[{'role': str, 'content': str}]]
    
    Returns:
        list[str] decoded new tokens only
    """
    enc = tokenizer.apply_chat_template(
        message_histories,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = 'pt',
        padding = True,
        truncation = True,
        max_length = MAX_INPUT_TOKENS,
        return_dict = True
    )

    input_ids = enc['input_ids'].to(main_device)
    attention_mask = enc['attention_mask'].to(main_device)
    prompt_len = input_ids.shape[1] # IMPORTANT: with padding, prompt_len is the padded length, so new tokens start here for all rows

    out = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        max_new_tokens = MAX_NEW_TOKENS,
        do_sample = True,
        temperature = TEMPERATURE,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
        use_cache = True,
    )

    new_tokens = out[:, prompt_len:] # (batch, <=max_new_tokens) padded on the right

    assistant_texts, state_texts = [], []

    for i in range(out.shape[0]):
        prompt_ids = input_ids[i][attention_mask[i].bool()].tolist() # Remove lpad from the prompt using the attention mask

        gen_ids = new_tokens[i].tolist()
        # Strip right-pad tokens added by batching different gen lengths
        while gen_ids and gen_ids[-1] == tokenizer.pad_token_id:
            gen_ids.pop()

        # Assistant text used for subsequent rounds (no special tokens)
        asst = tokenizer.decode(gen_ids, skip_special_tokens = True).strip()
        asst = asst if asst else None

        # Full composite prompt + output (keep special tokens / instruct formatting)
        state_text = tokenizer.decode(prompt_ids + gen_ids, skip_special_tokens = False)

        assistant_texts.append(asst)
        state_texts.append(state_text)

    return assistant_texts, state_texts


def generate_convs_data_local(user_queries_df, target_dir):
    os.makedirs(target_dir, exist_ok = True)
    out_path = os.path.join(target_dir, model_prefix + '-raw.csv')

    print(f"\n{'='*50}")
    print(f"Local generation model: {model_prefix}")

    responses = {}
    max_rounds = int(user_queries_df['user_query_ix'].max()) + 1

    for round_ix in range(max_rounds):
        print(f"Round {round_ix}...")

        current_round_df = user_queries_df[user_queries_df["user_query_ix"] == round_ix]
        message_histories = []
        conv_ids = []
        skipped = 0

        for _, row in current_round_df.iterrows():
            conv_id = int(row["conv_id"])

            # Skip if any prior round missing
            prior_failed = any(responses.get((conv_id, prev_ix), {}).get("assistant") is None for prev_ix in range(round_ix))
            if prior_failed:
                skipped += 1
                responses[(conv_id, round_ix)] = {"assistant": None, "state_text": None}
                continue

            # Prev turns
            msgs = []
            for prev_ix in range(round_ix):
                prev_user = user_queries_df[(user_queries_df['conv_id'] == conv_id) & (user_queries_df["user_query_ix"] == prev_ix)]['user_query'].item()
                msgs.append({"role": "user", "content": prev_user})
                prev_asst = responses[(conv_id, prev_ix)]["assistant"]
                msgs.append({"role": "assistant", "content": prev_asst})
                
            # Current user turn
            msgs.append({"role": "user", "content": row["user_query"]})
            message_histories.append(msgs)
            conv_ids.append(conv_id)

        if not message_histories:
            print(f"  All skipped ({skipped}) due to prior failures")
            continue

        all_asst = []
        all_state = []
        for batch in tqdm(chunk_list(message_histories, BATCH_SIZE)):
            batch_asst, batch_state = generate_local_batch(batch)
            all_asst.extend(batch_asst)
            all_state.extend(batch_state)

        for conv_id, asst_text, state_text in zip(conv_ids, all_asst, all_state):
            responses[(conv_id, round_ix)] = {"assistant": asst_text, "state_text": state_text}

        print(f"  Completed: {len(conv_ids)} convs, skipped: {skipped}")
    
    responses_df = pd.DataFrame(
        [{"conv_id": cid, "user_query_ix": qix, **resp}
        for (cid, qix), resp in responses.items()]
    )

    model_results_df = (
        user_queries_df
        .merge(responses_df, on = ["conv_id", "user_query_ix"], how="left")
        .assign(model = model_prefix, model_prefix = model_prefix)
    )

    model_results_df.to_csv(out_path, index = False)
    return model_results_df

from local_runtime import generate_local_batch
local_result_df = generate_convs_data_local(
    user_queries_df,
    target_dir = f"{ws}/experiments/role-analysis/data/conversations",
)


Local generation model: gptoss-20b
Round 0...


  0%|          | 0/4 [00:00<?, ?it/s]

Reusing generation batch: ea4e5dd3ef 16


Generation batch b88c9018cc: 250/4000 steps, 39.3s, batch 64


Generation batch b88c9018cc: 500/4000 steps, 54.4s, batch 64


Generation batch b88c9018cc: 750/4000 steps, 70.7s, batch 64


... (184 outputs left out)

Generated local batch: 2 seconds 187.6 tokens 5714 complete 2


100%|██████████| 2/2 [31:43<00:00, 919.77s/it] 

100%|██████████| 2/2 [31:43<00:00, 951.60s/it]

  Completed: 24 convs, skipped: 1


In [10]:
# Parse Harmony channels directly: the original tail(1) discarded all CoT segments.
from local_runtime import parse_latest_response, atomic_json, OUT
raw_df = local_result_df.copy()
pairs = [parse_latest_response(s) if isinstance(s,str) else (None,None) for s in raw_df['state_text']]
raw_df['cot'] = [p[0] for p in pairs]
raw_df['parsed_final'] = [p[1] for p in pairs]
failed = raw_df.loc[raw_df[['assistant','cot','parsed_final']].isna().any(axis=1), 'conv_id'].unique()
print('Dropping incomplete conversations:',len(failed))


Dropping incomplete conversations: 16


In [11]:
merged = (raw_df[~raw_df['conv_id'].isin(failed)]
    .drop(columns=['state_text','assistant']).rename(columns={'parsed_final':'assistant'}))
merged.to_csv(f'{ws}/experiments/role-analysis/data/conversations/{model_prefix}.csv',index=False)
atomic_json(OUT/'conversation_generation.json', {
    'requested_conversations':int(user_queries_df.conv_id.nunique()),
    'requested_user_turns':len(user_queries_df), 'complete_conversations':int(merged.conv_id.nunique()),
    'complete_user_turns':len(merged), 'dropped_incomplete_conversations':len(failed),
    'cross_model_comparison':'skipped by user instruction',
})
display(merged.groupby('dataset').agg(conversations=('conv_id','nunique')))


,conversations
dataset,
oasst,92
toxicchat,92
